In [ ]:
import pandas as pd

# Read and display column names for each data file
try:
    antidepressants_df = pd.read_csv('1.Antidepressants_drug_list.csv', nrows=5)
    print('1.Antidepressants_drug_list.csv columns:', antidepressants_df.columns.tolist())
except Exception as e:
    print('Error reading 1.Antidepressants_drug_list.csv:', e)

try:
    adverse_dict_df = pd.read_csv('2.adverse_reactions_to_organ_system.csv', nrows=5)
    print('2.adverse_reactions_to_organ_system.csv columns:', adverse_dict_df.columns.tolist())
except Exception as e:
    print('Error reading 2.adverse_reactions_to_organ_system.csv:', e)

try:
    reddit_df = pd.read_csv('3.combined_preprocessed.csv', nrows=5)
    print('3.combined_preprocessed.csv columns:', reddit_df.columns.tolist())
except Exception as e:
    print('Error reading 3.combined_preprocessed.csv:', e)

try:
    faers_df = pd.read_csv('4.FAERS_data.csv', nrows=5)
    print('FAERS_data.csv columns:', faers_df.columns.tolist())
except Exception as e:
    print('Error reading FAERS_data.csv:', e)

# Adverse Event Detection and Analysis

This notebook follows the methodology described in `task.md` to analyze adverse events (AEs) related to antidepressant drugs using Reddit and FAERS data. Each step is clearly separated and commented for clarity.

## 1. Data Loading

Load all required datasets: Reddit data, drug list, adverse reactions dictionary, and FAERS data.

In [ ]:
import pandas as pd

# Load Reddit data
reddit_df = pd.read_csv('3.combined_preprocessed.csv')
# Load drug list (corrected filename)
antidepressants_df = pd.read_csv('1.Antidepressants_drug_list.csv')
# Load adverse reactions dictionary
adverse_dict_df = pd.read_csv('2.adverse_reactions_to_organ_system.csv')
# Load FAERS data
faers_df = pd.read_csv('4.FAERS_data.csv')

# Display the first few rows of each dataset for verification
reddit_df.head(), antidepressants_df.head(), adverse_dict_df.head(), faers_df.head()

## 2. Dictionary-Based AE Extraction

Preprocess Reddit text and extract adverse events (AEs) using the dictionary. Map AEs to organ systems and extract drug mentions.

In [ ]:
import re

# Preprocess text: lowercasing, remove special chars, tokenize
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return tokens

# Prepare AE and drug dictionaries using correct column names
adverse_terms = adverse_dict_df['Adverse reactions'].str.lower().unique()
organ_map = dict(zip(adverse_dict_df['Adverse reactions'].str.lower(), adverse_dict_df['Organ System']))
drugs = pd.concat([
    antidepressants_df['Generic Name'].str.lower(),
    antidepressants_df['Brand Name(s) (US/UK/Other)'].str.lower()
]).dropna().unique()
drug_category_map = dict(zip(antidepressants_df['Generic Name'].str.lower(), antidepressants_df['Category']))

drug_ae_records = []

# Use correct Reddit columns: try to infer, fallback to 'combined_text' and 'row_id' if not found
reddit_text_col = 'combined_text' if 'combined_text' in reddit_df.columns else reddit_df.columns[0]
reddit_rowid_col = 'row_id' if 'row_id' in reddit_df.columns else reddit_df.columns[0]

for idx, row in reddit_df.iterrows():
    tokens = preprocess_text(row[reddit_text_col])
    found_aes = set([ae for ae in adverse_terms if ae in ' '.join(tokens)])
    found_drugs = set([drug for drug in drugs if drug in ' '.join(tokens)])
    for drug in found_drugs:
        for ae in found_aes:
            drug_ae_records.append({
                'row_id': row[reddit_rowid_col],
                'drug': drug,
                'ae': ae,
                'organ_system': organ_map.get(ae, 'Unknown'),
                'category': drug_category_map.get(drug, 'Unknown')
            })

# Convert to DataFrame
drug_ae_df = pd.DataFrame(drug_ae_records)
# Remove duplicate records
drug_ae_df = drug_ae_df.drop_duplicates()
drug_ae_df.to_csv('step2_drug_ae_extracted.csv', index=False)
drug_ae_df.head()

## 4. Descriptive Analysis

Analyze the frequency of AEs (overall, by organ system, by drug, by drug category). Show top AEs and drugs.

In [ ]:
# Step 3: Load extracted drug-AE data and visualize
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Read the large CSV efficiently
drug_ae_df = pd.read_csv('step2_drug_ae_extracted.csv', low_memory=False)
print('Columns:', drug_ae_df.columns.tolist())
print('Shape:', drug_ae_df.shape)
display(drug_ae_df.head())

# Dynamically detect column names (case-insensitive)
def get_col(df, name):
    for col in df.columns:
        if col.strip().lower() == name.lower():
            return col
    raise ValueError(f'Column {name} not found!')

# Get column names for AE, Drug, Organ System, Row ID
ae_col = get_col(drug_ae_df, 'ae')
drug_col = get_col(drug_ae_df, 'drug')
rowid_col = get_col(drug_ae_df, 'row_id')
organ_col = None
for c in drug_ae_df.columns:
    if c.strip().lower() in ['organ_system', 'organ system', 'organ']:
        organ_col = c
        break

# Plot top 20 most frequent Adverse Events (AEs) by unique post count (row_id) - log transformed
ae_counts = drug_ae_df.groupby(ae_col)[rowid_col].nunique().sort_values(ascending=False).head(20)
ae_counts_log = np.log1p(ae_counts)
plt.figure(figsize=(10,6))
sns.barplot(x=ae_counts_log.values, y=ae_counts_log.index, palette='viridis')
plt.xlabel('Log(Number of Unique Posts + 1)')
plt.ylabel('Adverse Event (AE)')
plt.title('Top 20 AEs: Unique Post Count (Log Transformed)')
plt.tight_layout()
plt.show()

# Plot top 20 most mentioned Drugs - log transformed
drug_counts = drug_ae_df[drug_col].value_counts().head(20)
drug_counts_log = np.log1p(drug_counts)
plt.figure(figsize=(10,6))
sns.barplot(x=drug_counts_log.values, y=drug_counts_log.index, palette='magma')
plt.xlabel('Log(Count + 1)')
plt.ylabel('Drug')
plt.title('Top 20 Most Mentioned Drugs (Log Transformed)')
plt.tight_layout()
plt.show()

# Plot top 20 most affected Organ Systems (if present) - log transformed
if organ_col is not None:
    organ_counts = drug_ae_df[organ_col].value_counts().head(20)
    organ_counts_log = np.log1p(organ_counts)
    plt.figure(figsize=(10,6))
    sns.barplot(x=organ_counts_log.values, y=organ_counts_log.index, palette='cubehelix')
    plt.xlabel('Log(Count + 1)')
    plt.ylabel('Organ System')
    plt.title('Top 20 Most Affected Organ Systems (Log Transformed)')
    plt.tight_layout()
    plt.show()
else:
    print('Column for Organ System not found in the data.')


## 5. Co-Occurrence Analysis

Analyze co-occurrence of drugs and AEs. Create a frequency matrix and visualize as a heatmap.

In [ ]:
# Drug-AE co-occurrence matrix
co_matrix = pd.crosstab(drug_ae_df['drug'], drug_ae_df['ae'])
co_matrix.to_csv('step4_drug_ae_cooccurrence.csv')
plt.figure(figsize=(12,8))
sns.heatmap(co_matrix, cmap='viridis', cbar_kws={'label': 'Co-occurrence Count'})
plt.title('Drug-AE Co-occurrence Heatmap (Dictionary-Based)')
plt.xlabel('Adverse Event')
plt.ylabel('Drug')
plt.show()

# Drug co-occurrence (pairs of drugs mentioned together)
from itertools import combinations
from collections import Counter

drug_pairs = []
for row_id, group in drug_ae_df.groupby('row_id'):
    drugs_in_post = group['drug'].unique()
    for pair in combinations(sorted(drugs_in_post), 2):
        drug_pairs.append(pair)

pair_counts = Counter(drug_pairs)
pair_df = pd.DataFrame(pair_counts.most_common(), columns=['drug_pair', 'count'])
pair_df['log_count'] = np.log1p(pair_df['count'])
pair_df.to_csv('step6_drug_pair_cooccurrence.csv', index=False)
pair_df.head(10)

## Drug Category Co-Occurrence Heatmap

Visualize the co-occurrence of drug categories (e.g., SSRI, SNRI) in Reddit posts as a heatmap.

In [ ]:
import numpy as np
# Create a mapping from drug to category
category_map = dict(zip(antidepressants_df['Generic Name'].str.lower(), antidepressants_df['Category']))

# Add category column to drug_ae_df if not present
if 'category' not in drug_ae_df.columns:
    drug_ae_df['category'] = drug_ae_df['drug'].map(category_map).fillna('Unknown')

# For each post, get unique categories mentioned
row_category = drug_ae_df.groupby('row_id')['category'].unique()

# Count co-occurrences
from itertools import combinations
from collections import Counter
cat_pairs = []
for cats in row_category:
    cats = [c for c in set(cats) if c != 'Unknown']
    for pair in combinations(sorted(cats), 2):
        cat_pairs.append(pair)
cat_pair_counts = Counter(cat_pairs)

# Create a matrix for heatmap
categories = sorted(set([c for pair in cat_pair_counts for c in pair]))
cat_matrix = np.zeros((len(categories), len(categories)), dtype=int)
cat_idx = {cat: i for i, cat in enumerate(categories)}
for (cat1, cat2), count in cat_pair_counts.items():
    i, j = cat_idx[cat1], cat_idx[cat2]
    cat_matrix[i, j] = count
    cat_matrix[j, i] = count  # symmetric

# Save the original (non-log) co-occurrence matrix to CSV
import pandas as pd
cat_matrix_df = pd.DataFrame(cat_matrix, index=categories, columns=categories)
cat_matrix_df.to_csv('step5_drug_category_cooccurrence.csv')

# Log transform for visualization (add 1 to avoid log(0))
cat_matrix_log = np.log1p(cat_matrix)

plt.figure(figsize=(8,6))
sns.heatmap(cat_matrix_log, xticklabels=categories, yticklabels=categories, cmap='Blues', annot=True, fmt='.2f')
plt.title('Log-Transformed Drug Category Co-Occurrence Heatmap')
plt.xlabel('Drug Category')
plt.ylabel('Drug Category')
plt.show()


## 7. Validation with FAERS

Aggregate Reddit AE counts by drug and reaction, compare with FAERS, and identify underreported AEs.

In [ ]:
# Aggregate Reddit AE counts by drug and reaction (using dynamic column names)
def get_col(df, name):
    for col in df.columns:
        if col.strip().lower() == name.lower():
            return col
    raise ValueError(f'Column {name} not found!')

drug_col = get_col(drug_ae_df, 'drug')
ae_col = get_col(drug_ae_df, 'ae')

# Use dynamic columns for groupby
reddit_counts = drug_ae_df.groupby([drug_col, ae_col]).size().reset_index(name='reddit_count')
reddit_counts.to_csv('step7_drug_ae_counts.csv', index=False)

# Prepare FAERS data for comparison (dynamic columns)
faers_drug_col = get_col(faers_df, 'drug')
faers_ae_col = None
for c in faers_df.columns:
    if c.strip().lower() in ['adverse reaction', 'adverse_reaction', 'ae']:
        faers_ae_col = c
        break
faers_case_col = None
for c in faers_df.columns:
    if c.strip().lower() in ['case count', 'case_count', 'count']:
        faers_case_col = c
        break

# Standardize to string for merge (handles e.g. '2014 ANXIETY' vs 'ANXIETY')
faers_counts = faers_df.copy()
faers_counts[faers_drug_col] = faers_counts[faers_drug_col].astype(str).str.strip().str.lower()
faers_counts[faers_ae_col] = faers_counts[faers_ae_col].astype(str).str.strip().str.lower()
faers_counts = faers_counts.groupby([faers_drug_col, faers_ae_col])[faers_case_col].sum().reset_index()
faers_counts.rename(columns={faers_ae_col: 'ae', faers_drug_col: 'drug', faers_case_col: 'faers_count'}, inplace=True)

reddit_counts['drug'] = reddit_counts['drug'].astype(str).str.strip().str.lower()
reddit_counts['ae'] = reddit_counts['ae'].astype(str).str.strip().str.lower()

# Try partial match if exact match fails (e.g. '2014 ANXIETY' contains 'anxiety')
merged = pd.merge(reddit_counts, faers_counts, on=['drug', 'ae'], how='left')

# If faers_count is NaN, try to find a partial match in FAERS AE for that drug
import numpy as np
for idx, row in merged[merged['faers_count'].isna()].iterrows():
    faers_subset = faers_counts[faers_counts['drug'] == row['drug']]
    # Try to find a partial match (Reddit AE in FAERS AE)
    match = faers_subset[faers_subset['ae'].str.contains(row['ae'], case=False, na=False)]
    if not match.empty:
        merged.at[idx, 'faers_count'] = match['faers_count'].sum()
    else:
        # Try the other way around (FAERS AE in Reddit AE)
        match = faers_subset[faers_subset['ae'].apply(lambda x: row['ae'] in x if isinstance(x, str) else False)]
        if not match.empty:
            merged.at[idx, 'faers_count'] = match['faers_count'].sum()

merged['faers_count'] = merged['faers_count'].fillna(0)
merged.to_csv('step7_reddit_faers_comparison.csv', index=False)

# Show some merged results for inspection
print(merged.sort_values('reddit_count', ascending=False).head(20))

# Identify underreported AEs: frequent in Reddit, rare in FAERS (e.g., Reddit count > 5, FAERS count == 0)
underreported = merged[(merged['reddit_count'] > 5) & (merged['faers_count'] == 0)]
underreported = underreported.sort_values(by='reddit_count', ascending=False)
underreported.head(10)


In [ ]:
# Debug: Show all FAERS AEs for citalopram
print('FAERS AEs for citalopram:')
print(faers_counts[faers_counts['drug'] == 'citalopram'])

# Debug: Show all Reddit AEs for citalopram
print('Reddit AEs for citalopram:')
print(reddit_counts[reddit_counts['drug'] == 'citalopram'])

# After merge, show merged rows for citalopram and anxiety
print('Merged rows for citalopram and anxiety:')
print(merged[(merged['drug'] == 'citalopram') & (merged['ae'].str.contains('anxiety'))])

In [ ]:
# Comparison barplot: log-transformed Reddit vs FAERS counts for top 20 AEs where Reddit has edge
# For each AE, sum Reddit counts across all drugs (not per-drug)
# Group by AE, sum reddit_count and faers_count

ae_sum = merged.groupby('ae')[['reddit_count', 'faers_count']].sum().reset_index()
ae_sum['reddit_minus_faers'] = ae_sum['reddit_count'] - ae_sum['faers_count']
# Top 20 AEs where Reddit has the largest edge

top20 = ae_sum.sort_values('reddit_minus_faers', ascending=False).head(20)

plt.figure(figsize=(12,7))
bar_width = 0.4
indices = np.arange(len(top20))
plt.bar(indices, np.log1p(top20['reddit_count']), bar_width, label='Reddit (log)', color='tab:blue')
plt.bar(indices + bar_width, np.log1p(top20['faers_count']), bar_width, label='FAERS (log)', color='tab:orange')
plt.xticks(indices + bar_width/2, top20['ae'], rotation=45, ha='right')
plt.ylabel('Log(Count + 1)')
plt.xlabel('Adverse Event (AE)')
plt.title('Top 20 AEs: Log-Transformed Reddit vs FAERS Counts (Reddit Edge, Summed Across Drugs)')
plt.legend()
plt.tight_layout()
plt.show()


## What is Log Transformation? (Simple Explanation)

Imagine you have a list of numbers, and some are really big while others are small. If you try to make a bar chart, the big numbers make the small ones look tiny, and it's hard to see the differences.

A log transformation is like a "shrinking tool" for numbers. It makes big numbers smaller, but keeps small numbers almost the same. This way, you can see patterns and compare things more easily on a chart.

For example:
- Without log: 1, 10, 100, 1000 (the bars for 100 and 1000 are huge!)
- With log: log(1)=0, log(10)=1, log(100)=2, log(1000)=3 (now the bars are closer in size)

So, log transformation helps us see differences between values, even when some are much bigger than others. It's super useful for making charts easier to understand!

#### Longititudinal Data

In [ ]:
# Create a longitudinal arrangement of drug_ae_df with created_date, author, subreddit, combined_text, url from reddit_df

# Columns: row_id, created_date, author, subreddit, combined_text, url, drug, ae, organsystem, category

# Remove rows where author is '[deleted]', remove duplicates, and save to CSV


def get_col(df, name):

    for col in df.columns:

        if col.strip().lower() == name.lower():

            return col

    raise ValueError(f'Column {name} not found!')



rowid_col = get_col(drug_ae_df, 'row_id')

drug_col = get_col(drug_ae_df, 'drug')

ae_col = get_col(drug_ae_df, 'ae')

organ_col = None

for c in drug_ae_df.columns:

    if c.strip().lower() in ['organ_system', 'organ system', 'organ']:

        organ_col = c

        break

category_col = get_col(drug_ae_df, 'category') if 'category' in [c.lower() for c in drug_ae_df.columns] else None


# Get relevant columns from reddit_df if present

reddit_rowid_col = get_col(reddit_df, 'row_id')

created_date_col = None

author_col = None

subreddit_col = None

combined_text_col = None

url_col = None

for c in reddit_df.columns:

    if 'created_date' in c.lower():

        created_date_col = c

    if c.strip().lower() == 'author':

        author_col = c

    if c.strip().lower() == 'subreddit':

        subreddit_col = c

    if c.strip().lower() == 'combined_text':

        combined_text_col = c

    if c.strip().lower() == 'url':

        url_col = c


# Merge drug_ae_df with reddit_df to get all relevant columns

merge_cols = [rowid_col, drug_col, ae_col]

if organ_col: merge_cols.append(organ_col)

if category_col: merge_cols.append(category_col)



longitudinal_df = drug_ae_df[merge_cols].drop_duplicates()


# Only merge if columns exist in reddit_df

if any([created_date_col, author_col, subreddit_col, combined_text_col, url_col]):

    merge_fields = [reddit_rowid_col]

    if created_date_col: merge_fields.append(created_date_col)

    if author_col: merge_fields.append(author_col)

    if subreddit_col: merge_fields.append(subreddit_col)

    if combined_text_col: merge_fields.append(combined_text_col)

    if url_col: merge_fields.append(url_col)

    # Remove author_flair_text if present

    merge_fields = [f for f in merge_fields if f.lower() != 'author_flair_text']

    reddit_meta = reddit_df[merge_fields].drop_duplicates()

    # Merge on row_id

    longitudinal_df = longitudinal_df.merge(reddit_meta, left_on=rowid_col, right_on=reddit_rowid_col, how='left')

    # Remove rows where author is '[deleted]'

    if author_col and author_col in longitudinal_df.columns:

        longitudinal_df = longitudinal_df[longitudinal_df[author_col] != '[deleted]']

    # Remove duplicates again after merge

    longitudinal_df = longitudinal_df.drop_duplicates()

    # Reorder columns

    out_cols = [rowid_col]

    if created_date_col: out_cols.append(created_date_col)

    if author_col: out_cols.append(author_col)

    if subreddit_col: out_cols.append(subreddit_col)

    if combined_text_col: out_cols.append(combined_text_col)

    if url_col: out_cols.append(url_col)

    out_cols += [drug_col, ae_col]

    if organ_col: out_cols.append(organ_col)

    if category_col: out_cols.append(category_col)

    longitudinal_df = longitudinal_df[out_cols]



longitudinal_df = longitudinal_df.sort_values(by=[rowid_col, drug_col, ae_col])

longitudinal_df.to_csv('drug_ae_longitudinal.csv', index=False)

longitudinal_df.head()

In [ ]:
# Timeline plot: poster count by drug category (SSRI, SNRI, etc) from Jan 1st 2015 to Dec 31st 2025
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter
date_col = None
for col in reddit_df.columns:
    if 'date' in col.lower():
        date_col = col
        break
if date_col is None:
    raise ValueError('No date column found in Reddit data.')
# Parse dates
reddit_df[date_col] = pd.to_datetime(reddit_df[date_col], errors='coerce')
# Filter date range
start_date = pd.Timestamp('2015-01-01')
end_date = pd.Timestamp('2025-12-31')
reddit_df = reddit_df[(reddit_df[date_col] >= start_date) & (reddit_df[date_col] <= end_date)]
# Prepare drug category mapping
category_map = dict(zip(antidepressants_df['Generic Name'].str.lower(), antidepressants_df['Category']))
# Extract drug mentions and categories per post
def extract_categories(text):
    text = str(text).lower()
    found_drugs = set([drug for drug in category_map.keys() if drug in text])
    found_categories = set([category_map[drug] for drug in found_drugs if category_map[drug] != 'Unknown'])
    return list(found_categories) if found_categories else ['Unknown']
reddit_df['categories'] = reddit_df['combined_text'].apply(extract_categories)
# Explode categories for counting
timeline_df = reddit_df.explode('categories')
# Group by month and category
timeline_df['month'] = timeline_df[date_col].dt.to_period('M')
timeline_counts = timeline_df.groupby(['month','categories']).size().reset_index(name='poster_count')
# Pivot for plotting
timeline_pivot = timeline_counts.pivot(index='month', columns='categories', values='poster_count').fillna(0)
# Plot
plt.figure(figsize=(16,8))
timeline_pivot.plot(ax=plt.gca(), linewidth=3)
plt.xlabel('Month')
plt.ylabel('Poster Count')
plt.title('Timeline of Poster Count by Drug Category (2015-2025)')
plt.legend(title='Drug Category')
plt.tight_layout()
plt.show()

In [ ]:
# Annotate Reddit timeline with major events impacting antidepressant usage/reporting (e.g., COVID-19 pandemic)
import matplotlib.dates as mdates

# --- Define major events ---
events = [
    {'name': 'COVID-19 Pandemic Declared', 'date': pd.Timestamp('2020-03-11')},
    {'name': 'First US Lockdown', 'date': pd.Timestamp('2020-03-15')},
    {'name': 'COVID Vaccine Rollout', 'date': pd.Timestamp('2021-01-01')},
    # Add more events as needed
    # Example: {'name': 'CDC Guidance Update', 'date': pd.Timestamp('2020-04-03')},
    # Example: {'name': 'WHO Pandemic End', 'date': pd.Timestamp('2023-05-05')},
    # Example: {'name': 'Telehealth Expansion', 'date': pd.Timestamp('2020-04-01')},
]

# --- Plot Reddit timeline with event markers ---
plt.figure(figsize=(16,8))
timeline_pivot.plot(ax=plt.gca(), linewidth=3, alpha=0.7)

# Add vertical lines and labels for events
for event in events:
    plt.axvline(event['date'], color='red', linestyle='--', linewidth=2, alpha=0.8)
    plt.text(event['date'], plt.ylim()[1]*0.95, event['name'], color='red', rotation=90, va='top', ha='right', fontsize=12, backgroundcolor='white')

plt.xlabel('Month')
plt.ylabel('Poster Count')
plt.title('Timeline of Poster Count by Drug Category (Reddit, with Major Events Marked)')
plt.legend(title='Drug Category')
plt.tight_layout()
plt.show()

In [ ]:
# Annotate Reddit timeline with major events impacting antidepressant usage/reporting (COVID-19, telehealth, economic, etc.)
import matplotlib.dates as mdates

# --- Define major events ---
events = [
    {'name': 'COVID-19 Pandemic Declared', 'date': pd.Timestamp('2020-03-11')},
    {'name': 'First US Lockdown', 'date': pd.Timestamp('2020-03-15')},
    {'name': 'COVID Vaccine Rollout', 'date': pd.Timestamp('2021-01-01')},
    {'name': 'Telehealth Expansion', 'date': pd.Timestamp('2020-04-01')},
    {'name': 'CDC Guidance Update', 'date': pd.Timestamp('2020-04-03')},
    {'name': 'WHO Pandemic End', 'date': pd.Timestamp('2023-05-05')},
    {'name': 'US Economic Stimulus', 'date': pd.Timestamp('2020-03-27')},
    {'name': 'Global Mental Health Awareness Month', 'date': pd.Timestamp('2021-05-01')},
    {'name': 'US Election', 'date': pd.Timestamp('2020-11-03')},
    {'name': 'George Floyd Protests', 'date': pd.Timestamp('2020-05-25')},
    # Add more events as needed
]

# --- Plot Reddit timeline with event markers ---
plt.figure(figsize=(16,8))
timeline_pivot.plot(ax=plt.gca(), linewidth=3, alpha=0.7)

# Add vertical lines and labels for events
for event in events:
    plt.axvline(event['date'], color='red', linestyle='--', linewidth=2, alpha=0.8)
    plt.text(event['date'], plt.ylim()[1]*0.95, event['name'], color='red', rotation=90, va='top', ha='right', fontsize=12, backgroundcolor='white')

plt.xlabel('Month')
plt.ylabel('Poster Count')
plt.title('Timeline of Poster Count by Drug Category (Reddit, with Major Events Marked)')
plt.legend(title='Drug Category')
plt.tight_layout()
plt.show()

In [ ]:
# Deep statistical analysis of SSRIs peak (2023-2025) in Reddit timeline
import scipy.stats as stats

# 1. Extract SSRIs timeline from timeline_pivot
ssri_col = None
for col in timeline_pivot.columns:
    if 'ssri' in str(col).lower():
        ssri_col = col
        break
if ssri_col is None:
    print('No SSRI category found in timeline_pivot.')
else:
    ssri_timeline = timeline_pivot[ssri_col]

    # 2. Focus on 2023-2025 period
    ssri_2023_2025 = ssri_timeline[ssri_timeline.index >= '2023-01']
    ssri_2023_2025 = ssri_2023_2025[ssri_2023_2025.index <= '2025-12']

    # 3. Compare with previous years (2018-2022)
    ssri_2018_2022 = ssri_timeline[ssri_timeline.index >= '2018-01']
    ssri_2018_2022 = ssri_2018_2022[ssri_2018_2022.index <= '2022-12']

    # 4. Statistical test: Mann-Whitney U (non-parametric, robust to outliers)
    u_stat, p_value = stats.mannwhitneyu(ssri_2023_2025, ssri_2018_2022, alternative='greater')
    print(f'Mann-Whitney U test (2023-2025 vs 2018-2022): U={u_stat}, p={p_value:.4g}')

    # 5. Effect size: Cohen's d
    mean_2023_2025 = ssri_2023_2025.mean()
    mean_2018_2022 = ssri_2018_2022.mean()
    std_2023_2025 = ssri_2023_2025.std()
    std_2018_2022 = ssri_2018_2022.std()
    pooled_std = ((std_2023_2025**2 + std_2018_2022**2)/2)**0.5
    cohens_d = (mean_2023_2025 - mean_2018_2022) / pooled_std if pooled_std > 0 else float('nan')
    print(f"Cohen's d effect size: {cohens_d:.3f}")

    # 6. Visualize distributions
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,6))
    plt.hist(ssri_2018_2022, bins=12, alpha=0.6, label='2018-2022')
    plt.hist(ssri_2023_2025, bins=12, alpha=0.6, label='2023-2025')
    plt.xlabel('Monthly Poster Count (SSRIs)')
    plt.ylabel('Frequency')
    plt.title('SSRIs Poster Count Distribution: 2018-2022 vs 2023-2025')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 7. Summary statistics
    print('2018-2022 SSRIs: mean={:.1f}, std={:.1f}, n={}'.format(mean_2018_2022, std_2018_2022, len(ssri_2018_2022)))
    print('2023-2025 SSRIs: mean={:.1f}, std={:.1f}, n={}'.format(mean_2023_2025, std_2023_2025, len(ssri_2023_2025)))

## FAERS Timeline Plot by Drug Category
This cell visualizes the annual FAERS case counts for each antidepressant drug category.

In [ ]:
# Timeline plot: FAERS case count by drug category per year
import matplotlib.pyplot as plt
import seaborn as sns
# Ensure 'faers_df' is loaded and columns are correct
faers_df.columns = [c.strip() for c in faers_df.columns]
# Lowercase for merge
faers_df['Drug'] = faers_df['Drug'].astype(str).str.lower()
# Map drugs to categories
category_map = dict(zip(antidepressants_df['Generic Name'].str.lower(), antidepressants_df['Category']))
faers_df['Category'] = faers_df['Drug'].map(category_map).fillna('Unknown')
# Group by year and category
faers_timeline = faers_df.groupby(['Year', 'Category'])['Count'].sum().reset_index()
# Pivot for plotting
faers_pivot = faers_timeline.pivot(index='Year', columns='Category', values='Count').fillna(0)
# Plot
plt.figure(figsize=(16,8))
ax = plt.gca()
faers_pivot.plot(ax=ax, linewidth=3)
# Set x-axis ticks to every year interval
ax.set_xticks(faers_pivot.index)
ax.set_xticklabels(faers_pivot.index.astype(str), rotation=45)
plt.xlabel('Year')
plt.ylabel('FAERS Case Count')
plt.title('FAERS Timeline: Case Count by Drug Category (2014-2025)')
plt.legend(title='Drug Category')
plt.tight_layout()
plt.show()

## FAERS Timeline Plot for Each Drug by Category
The following cell creates a separate timeline plot for each drug category, with each line representing a drug in that category.

In [ ]:
# Timeline plot: FAERS case count for each drug, separate plot for every category
import matplotlib.pyplot as plt
import seaborn as sns
# Ensure 'faers_df' is loaded and columns are correct
faers_df.columns = [c.strip() for c in faers_df.columns]
faers_df['Drug'] = faers_df['Drug'].astype(str).str.lower()
category_map = dict(zip(antidepressants_df['Generic Name'].str.lower(), antidepressants_df['Category']))
faers_df['Category'] = faers_df['Drug'].map(category_map).fillna('Unknown')
# Get unique categories (excluding 'Unknown')
categories = sorted([c for c in faers_df['Category'].unique() if c != 'Unknown'])
for cat in categories:
    cat_df = faers_df[faers_df['Category'] == cat]
    # Pivot: index=Year, columns=Drug, values=Count
    pivot = cat_df.pivot_table(index='Year', columns='Drug', values='Count', aggfunc='sum').fillna(0)
    plt.figure(figsize=(16,8))
    pivot.plot(ax=plt.gca(), linewidth=2)
    plt.xlabel('Year')
    plt.ylabel('FAERS Case Count')
    plt.title(f'FAERS Timeline: Case Count by Drug ({cat})')
    plt.legend(title='Drug', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax = plt.gca()
    ax.set_xticks(pivot.index)
    ax.set_xticklabels(pivot.index.astype(str), rotation=45)
    plt.tight_layout()
    plt.show()

#### Sample Longititudinal data

In [ ]:
# Save a sample of 1000 records from the longitudinal data to a new CSV file, sorted by author and created_date
# Detect column names dynamically for author and created_date

author_col = None
created_date_col = None
for c in longitudinal_df.columns:
    if c.strip().lower() == 'author':
        author_col = c
    if 'created_date' in c.lower():
        created_date_col = c

longitudinal_sample = longitudinal_df.sample(n=min(1000, len(longitudinal_df)), random_state=42)
if author_col and created_date_col:
    longitudinal_sample = longitudinal_sample.sort_values(by=[author_col, created_date_col])
elif author_col:
    longitudinal_sample = longitudinal_sample.sort_values(by=author_col)
elif created_date_col:
    longitudinal_sample = longitudinal_sample.sort_values(by=created_date_col)

longitudinal_sample.to_csv('drug_ae_longitudinal_sample1000.csv', index=False)
longitudinal_sample.head()

In [ ]:
# Timeline plot: top 20 subreddits by post count (2015-2025) using 'created_date' and 'subreddit', yearly interval
import matplotlib.pyplot as plt
import pandas as pd

# Use reddit_df and ensure 'subreddit' and 'created_date' columns exist
if 'created_date' not in reddit_df.columns:
    raise ValueError("'created_date' column not found in reddit_df")
if 'subreddit' not in reddit_df.columns:
    raise ValueError("'subreddit' column not found in reddit_df")

df = reddit_df.copy()
if not pd.api.types.is_datetime64_any_dtype(df['created_date']):
    df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')

# Filter for 2015-2025
df = df[(df['created_date'].dt.year >= 2015) & (df['created_date'].dt.year <= 2025)]
df = df.dropna(subset=['subreddit'])

# Add a 'year' column for grouping
if 'year' not in df.columns:
    df['year'] = df['created_date'].dt.year

# Group by year and subreddit, count posts
if not df.empty:
    post_counts = df.groupby(['year', 'subreddit']).size().unstack(fill_value=0)
    post_counts.to_csv('subreddit_post_counts_by_year.csv')
    subreddit_totals = post_counts.sum(axis=0).sort_values(ascending=False)
    top20_subreddits = subreddit_totals.head(20).index.tolist()
    post_counts_top20 = post_counts[top20_subreddits]
    if not post_counts_top20.empty:
        plt.figure(figsize=(12, 6))
        ax = plt.gca()
        post_counts_top20.plot(ax=ax, marker='o')
        ax.set_xticks(post_counts_top20.index)
        ax.set_xticklabels(post_counts_top20.index.astype(str), rotation=45)
        plt.title('Top 20 Subreddits Post Count Timeline (2015-2025)')
        plt.xlabel('Year')
        plt.ylabel('Post Count')
        plt.legend(title='Subreddit', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
    else:
        print('No post counts to plot for the selected years.')
else:
    print('No data available for the selected years.')

In [ ]:
# Analyze which drug categories are discussed in the top 20 subreddits (2015-2025)
import pandas as pd

# Ensure 'reddit_df' is loaded and has the necessary columns
if 'created_date' not in reddit_df.columns:
    raise ValueError("'created_date' column not found in reddit_df")
if 'subreddit' not in reddit_df.columns:
    raise ValueError("'subreddit' column not found in reddit_df")

# Copy and filter for 2015-2025
cat_df = reddit_df.copy()
if not pd.api.types.is_datetime64_any_dtype(cat_df['created_date']):
    cat_df['created_date'] = pd.to_datetime(cat_df['created_date'], errors='coerce')
cat_df = cat_df[(cat_df['created_date'].dt.year >= 2015) & (cat_df['created_date'].dt.year <= 2025)]
cat_df = cat_df.dropna(subset=['subreddit'])

# Get top 20 subreddits by post count
subreddit_counts = cat_df['subreddit'].value_counts().head(20)
top20_subreddits = subreddit_counts.index.tolist()

# Prepare drug category mapping
category_map = dict(zip(antidepressants_df['Generic Name'].str.lower(), antidepressants_df['Category']))

def extract_categories(text):
    text = str(text).lower()
    found_drugs = set([drug for drug in category_map.keys() if drug in text])
    found_categories = set([category_map[drug] for drug in found_drugs if category_map[drug] != 'Unknown'])
    return list(found_categories) if found_categories else ['Unknown']

# Filter to top 20 subreddits
cat_df = cat_df[cat_df['subreddit'].isin(top20_subreddits)]
cat_df['categories'] = cat_df['combined_text'].apply(extract_categories)

# Explode categories for analysis
cat_exploded = cat_df.explode('categories')

# Group by subreddit and category, count posts
sub_cat_counts = cat_exploded.groupby(['subreddit', 'categories']).size().reset_index(name='post_count')

# For each subreddit, show the top discussed drug categories
result = sub_cat_counts.sort_values(['subreddit', 'post_count'], ascending=[True, False])

# Display the result
import pandas as pd
pd.set_option('display.max_rows', 60)
display(result)

# Optionally, save to CSV
result.to_csv('top20subreddits_drug_categories.csv', index=False)

In [ ]:
# Plot: Heatmap of drug categories discussed in top 20 subreddits (2015-2025)
import matplotlib.pyplot as plt
import seaborn as sns

# Pivot the result for heatmap: rows=subreddit, columns=category, values=post_count
heatmap_data = result.pivot(index='subreddit', columns='categories', values='post_count').fillna(0)

plt.figure(figsize=(16, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=0.5)
plt.title('Discussion of Drug Categories in Top 20 Subreddits (2015-2025)')
plt.xlabel('Drug Category')
plt.ylabel('Subreddit')
plt.tight_layout()
plt.show()

In [ ]:
# Show the number of records in reddit_df
total_records = len(reddit_df)
print(f'Total number of records in reddit_df: {total_records}')

In [ ]:
# Descriptive statistics for FAERS data
print('FAERS DataFrame shape:', faers_df.shape)
display(faers_df.describe(include='all'))

# Count of unique drugs, categories, and years
unique_drugs = faers_df['Drug'].nunique() if 'Drug' in faers_df.columns else None
unique_categories = faers_df['Category'].nunique() if 'Category' in faers_df.columns else None
unique_years = faers_df['Year'].nunique() if 'Year' in faers_df.columns else None
print(f"Unique drugs: {unique_drugs}")
print(f"Unique categories: {unique_categories}")
print(f"Unique years: {unique_years}")

# Top 10 drugs by total case count
if 'Drug' in faers_df.columns and 'Count' in faers_df.columns:
    top10_drugs = faers_df.groupby('Drug')['Count'].sum().sort_values(ascending=False).head(10)
    print('Top 10 drugs by total case count:')
    display(top10_drugs)

# Top 10 categories by total case count
if 'Category' in faers_df.columns and 'Count' in faers_df.columns:
    top10_categories = faers_df.groupby('Category')['Count'].sum().sort_values(ascending=False).head(10)
    print('Top 10 categories by total case count:')
    display(top10_categories)

# Years range
if 'Year' in faers_df.columns:
    print(f"Year range: {faers_df['Year'].min()} - {faers_df['Year'].max()}")

In [ ]:
# Plot: Top 20 drugs in FAERS by total case count
import matplotlib.pyplot as plt
import seaborn as sns

if 'Drug' in faers_df.columns and 'Count' in faers_df.columns:
    top20_drugs = faers_df.groupby('Drug')['Count'].sum().sort_values(ascending=False).head(20)
    plt.figure(figsize=(12, 6))
    sns.barplot(x=top20_drugs.values, y=top20_drugs.index, palette='crest')
    plt.xlabel('Total Case Count')
    plt.ylabel('Drug')
    plt.title('Top 20 Drugs in FAERS by Total Case Count')
    plt.tight_layout()
    plt.show()
else:
    print('Required columns not found in faers_df.')

In [ ]:
# Find authors with more than two records and save to CSV
import pandas as pd

# Read the required files
drug_ae_longitudinal = pd.read_csv('drug_ae_longitudinal.csv')

# Detect author column dynamically
author_col = None
for col in drug_ae_longitudinal.columns:
    if col.strip().lower() == 'author':
        author_col = col
        break
if author_col is None:
    raise ValueError('Author column not found in drug_ae_longitudinal.csv')

# Filter authors with more than two records
author_counts = drug_ae_longitudinal[author_col].value_counts()
keep_authors = author_counts[author_counts > 2].index
filtered_df = drug_ae_longitudinal[drug_ae_longitudinal[author_col].isin(keep_authors)]

# Save filtered data to CSV
filtered_df.to_csv('authors_more_than_two_records.csv', index=False)

# Show statistics
max_records = author_counts.max()
num_authors = len(author_counts)
num_authors_gt2 = len(keep_authors)
print(f'Maximum number of records by a single author: {max_records}')
print(f'Total number of unique authors: {num_authors}')
print(f'Number of authors with more than two records: {num_authors_gt2}')

In [ ]:
# Subset the author with the maximum number of records and save as CSV
# (Assumes previous cell loaded drug_ae_longitudinal and author_col)

# Find the author with the maximum number of records
max_author = author_counts.idxmax()
max_author_df = drug_ae_longitudinal[drug_ae_longitudinal[author_col] == max_author]

# Save to CSV
max_author_df.to_csv('max_author_records.csv', index=False)

print(f"Author with the most records: {max_author}")
print(f"Number of records: {len(max_author_df)}")

In [ ]:
# Subset the author with the maximum number of unique row_id records and save as CSV
# (Assumes previous cell loaded drug_ae_longitudinal and author_col)

# Detect row_id column dynamically
rowid_col = None
for col in drug_ae_longitudinal.columns:
    if col.strip().lower() == 'row_id':
        rowid_col = col
        break
if rowid_col is None:
    raise ValueError('row_id column not found in drug_ae_longitudinal.csv')

# Count unique row_id per author
author_rowid_counts = drug_ae_longitudinal.groupby(author_col)[rowid_col].nunique()
max_author = author_rowid_counts.idxmax()
max_count = author_rowid_counts.max()

# Subset for this author, keeping only one row per unique row_id
max_author_df = drug_ae_longitudinal[drug_ae_longitudinal[author_col] == max_author]
max_author_df = max_author_df.drop_duplicates(subset=[rowid_col])

# Save to CSV
max_author_df.to_csv('max_author_unique_rowid_records.csv', index=False)

print(f"Author with the most unique records: {max_author}")
print(f"Number of unique row_id records: {max_count}")